In [1]:
!pip install mne

     |████████████████████████████████| 6.6MB 3.2MB/s 


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from scipy.io import loadmat
import mne
from mne.decoding import CSP
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [3]:
def band_pass_filter(eeg, freq_range):
  sfreq = 250
  info = mne.create_info(22, 250, ch_types=["eeg"] * 22)
  raw = mne.io.RawArray(eeg.T, info)
  iir_params = dict(order=5, ftype='cheby2', rs=2.)
  # iir_params.update(ftype='cheby2',rp=1.,  # dB of acceptable pass-band ripple)
  # x = raw._data.T
  # filt = mne.filter.create_filter(x, sfreq, l_freq=freq_range[0], h_freq=freq_range[1],
  #                               method='iir', iir_params=iir_params,
  #                               verbose=True)
  raw = raw.filter(freq_range[0], freq_range[1], fir_design='firwin', method='iir', iir_params=iir_params)

  return raw._data.T
  # return filt

def load_data(mode='train', fno = 1):
  if fno != 4:
    if (mode=='train'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'T.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 3+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)

    elif (mode=='test'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'E.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 3+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)
      
    return df

  else:
    if (mode=='train'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'T.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 1+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)

    elif (mode=='test'):
      fname = '/content/drive/My Drive/BCI/A0' + str(fno) + 'E.mat'
      file_data = loadmat(fname)
      data = file_data['data']
      df = pd.DataFrame()
      for i in range(0, 6):
          idx = 1+i
          pos_data = data[0][idx][0][0][1]
          label_data = data[0][idx][0][0][2]
          temp = pd.DataFrame(data[0][idx][0][0][0])
          label = np.zeros(len(temp))
          count = 0
          for j in pos_data:
              label[j] = label_data[count]
              count += 1
          temp['class'] = label
          df = pd.concat([df, temp], ignore_index=True)
      
    return df

In [4]:
def drop_classes(df):
  i = 0
  indexes_to_drop = []
  while i < len(df):
    if(df['class'][i]==4):
      list2 = list(range(i, 1001+i))
      i += 1001
      indexes_to_drop.extend(list2)
    elif(df['class'][i]==3):
      list2 = list(range(i, 1001+i))
      i += 1001
      indexes_to_drop.extend(list2)
    else:
      i += 1

  indexes_to_keep = set(range(df.shape[0])) - set(indexes_to_drop)
  df_sliced = df.take(list(indexes_to_keep))

  df_sliced = df_sliced.reset_index(drop=True)
  return df_sliced

In [5]:
def fbcsp(df, sfreq, train=True, csp_objects=None):
  freq = 4
  increment = 4
  end_freq = 40
  event_dict = {'Left/Hands': 1, 'Right/Hands': 2}
  if train==True:
    csp_objects = []
    csp_data = []
    while freq < end_freq:
      freq_range = []
      freq_range.append(freq)
      freq_range.append(freq+increment)
      freq += increment
      out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

      df_new = pd.DataFrame(out_data)
      df_new['label'] = df.iloc[:, -1].values

      info = mne.create_info(23, sfreq, ch_types=["eeg"] * 22 + ['stim'] * 1)
      raw = mne.io.RawArray(df_new.T, info)

      events = mne.find_events(raw, stim_channel='22')
      picks = mne.pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False,
                   exclude='bads')
      epochs = mne.Epochs(raw, events, event_id=event_dict, tmin=-1, tmax=4, picks=picks, preload=True)
      epochs = epochs.crop(tmin=0.5, tmax=2.5)
      y = epochs.events[:, -1]
      X = epochs.get_data()

      csp = CSP(n_components=2, reg=None, log=True, norm_trace=False)

      final_data = csp.fit_transform(X, y)

      csp_objects.append(csp)
      csp_data.append(final_data)

    return np.array(csp_objects), np.array(csp_data), y
  
  else:
    csp_data = []
    count = 0
    while freq < end_freq:
      freq_range = []
      freq_range.append(freq)
      freq_range.append(freq+increment)
      freq += increment
      out_data = band_pass_filter(df.iloc[:, :-1].values, freq_range=freq_range)

      df_new = pd.DataFrame(out_data)
      df_new['label'] = df.iloc[:, -1].values

      info = mne.create_info(23, sfreq, ch_types=["eeg"] * 22 + ['stim'] * 1)
      raw = mne.io.RawArray(df_new.T, info)

      events = mne.find_events(raw, stim_channel='22')
      picks = mne.pick_types(raw.info, meg=False, eeg=True, stim=False, eog=False,
                   exclude='bads')
      epochs = mne.Epochs(raw, events, event_id=event_dict, tmin=-0.1, tmax=2, picks=picks, preload=True)
      epochs = epochs.crop(tmin=0.0, tmax=2.0)
      y = epochs.events[:, -1]
      X = epochs.get_data()

      final_data = csp_objects[count].transform(X)
      count += 1

      csp_data.append(final_data)

    return np.array(csp_data), y

  # return np.array(csp_objects), np.array(csp_data), y

In [6]:
def itr(n_class, p_class, c_time):
  B = (np.log2(n_class) + (p_class * np.log2(p_class)) + ((1-p_class) * np.log2((1-p_class)/(n_class-1)))) / c_time * 60

  return B

def performance_metrics(y_test, y_pred):
  acc = accuracy_score(y_test, y_pred)
  print('Accuracy Score: ', acc)
  print('Cohen Kappa Score: ', cohen_kappa_score(y_test, y_pred))
  print('ITR (bits per minute): ', itr(n_class=2, p_class=acc, c_time=2))
  print('Confusion Matrix: ', confusion_matrix(y_test, y_pred))

In [7]:
def get_train_data(fno):
  sfreq = 250
  trigger_points = {1:4, 2:4, 3:4, 4:4}

  df = load_data(mode='train', fno=fno)
  train_df = drop_classes(df)
  # train_df = df
  train_df = train_df.drop([22, 23, 24], axis=1)

  csp_objects, csp_data, y = fbcsp(train_df, sfreq, train=True)

  final_data = pd.DataFrame(csp_data[0])
  col_count = 4
  for i in range(1, len(csp_data)):
    for j in range(len(csp_data[i].T)):
      final_data[str(col_count)] = csp_data[i].T[j]
      col_count += 1

  return final_data.values, y, csp_objects

def get_eval_data(fno, csp_objects):
  sfreq = 250
  trigger_points = {1:4, 2:4, 3:4, 4:4}

  df = load_data(mode='test', fno=fno)
  train_df = drop_classes(df)
  # train_df = df
  train_df = train_df.drop([22, 23, 24], axis=1)
  csp_data, y = fbcsp(train_df, sfreq, train=False, csp_objects=csp_objects)

  final_data = pd.DataFrame(csp_data[0])
  col_count = 4
  for i in range(1, len(csp_data)):
    for j in range(len(csp_data[i].T)):
      final_data[str(col_count)] = csp_data[i].T[j]
      col_count += 1

  return final_data.values, y

In [8]:
def train_mibif(X, y):
  print('------Train---------')
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

  #get the best k features base on MIBIF algorithm
  select_K = SelectKBest(mutual_info_classif,k=8).fit(X, y)
  extra = select_K.get_support()
  if extra[0] == True:
    extra[1] = True
  for i in range(2, len(extra)):
    if extra[i] == True:
      if i%2 == 0:
        extra[i+1] = True
      else:
        extra[i-1] = True
  pos = np.where(extra==False)
  New_train = np.delete(X_train, list(pos[0]), 1)
  New_test = np.delete(X_test, list(pos[0]), 1)
  # New_train=select_K.transform(X_train)
  # New_test=select_K.transform(X_test)
  ss = StandardScaler()
  New_train = ss.fit_transform(New_train,y_train)
  New_test = ss.transform(New_test)

  print('####### SVM#####')
  svm = SVC()
  svm.fit(New_train, y_train)
  y_pred = svm.predict(New_test)
  performance_metrics(y_test, y_pred)
  
  print('##########LDA#########')
  lda = LinearDiscriminantAnalysis()
  lda.fit(New_train, y_train)
  y_pred = lda.predict(New_test)
  performance_metrics(y_test, y_pred)

  return list(pos[0]), svm, lda, ss

def eval_mibif(pos, svm, lda, ss, X, y):
  print('------Test--------')
  X = np.delete(X, pos, 1)
  X = ss.transform(X)
  print('#####SVM######')
  y_pred = svm.predict(X)
  performance_metrics(y, y_pred)
  print('#####LDA######')
  y_pred = lda.predict(X)
  performance_metrics(y, y_pred)


In [9]:
def train_rf(X, y):
  print('--------Train--------')
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
  clf = RandomForestClassifier(random_state=42)
  ss = StandardScaler()
  X_train = ss.fit_transform(X_train, y_train)
  clf.fit(X_train, y_train)
  X_test = ss.transform(X_test)
  y_pred = clf.predict(X_test)
  ####Random Forest#######
  performance_metrics(y_test, y_pred)

  return clf, ss
def eval_rf(clf, ss, X, y):
  print('--------Test------')
  X = ss.transform(X)
  y_pred = clf.predict(X)
  performance_metrics(y, y_pred)

# Subject A01

In [10]:
X, y, csp_objects = get_train_data(fno=1)
X_eval, y_eval = get_eval_data(fno=1,csp_objects=csp_objects)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 23 (2.2e-16 eps * 22 dim * 4.6e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channel

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 12 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 8.00, 12.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 11 (2.2e-16 eps * 22 dim * 2.2e+15  max singular val

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


    Using tolerance 11 (2.2e-16 eps * 22 dim * 2.2e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 12.00, 16.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 e

...

<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 36 - 40 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 36.00, 40.00 Hz: -4.00, -4.00 dB



<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped


<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


In [11]:
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)

------Train---------
####### SVM#####
Accuracy Score:  0.5555555555555556
Cohen Kappa Score:  0.11111111111111116
ITR (bits per minute):  0.2677182048533333
Confusion Matrix:  [[13  5]
 [11  7]]
##########LDA#########
Accuracy Score:  0.6388888888888888
Cohen Kappa Score:  0.2777777777777778
ITR (bits per minute):  1.6919510610185395
Confusion Matrix:  [[15  3]
 [10  8]]
------Test--------
#####SVM######
Accuracy Score:  0.4930555555555556
Cohen Kappa Score:  -0.01388888888888884
ITR (bits per minute):  0.004174599037647386
Confusion Matrix:  [[40 32]
 [41 31]]
#####LDA######
Accuracy Score:  0.5486111111111112
Cohen Kappa Score:  0.09722222222222221
ITR (bits per minute):  0.20487223857243664
Confusion Matrix:  [[47 25]
 [40 32]]


In [12]:
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

--------Train--------
Accuracy Score:  0.6388888888888888
Cohen Kappa Score:  0.2777777777777778
ITR (bits per minute):  1.6919510610185395
Confusion Matrix:  [[15  3]
 [10  8]]
--------Test------
Accuracy Score:  0.5277777777777778
Cohen Kappa Score:  0.05555555555555558
ITR (bits per minute):  0.06682583730054037
Confusion Matrix:  [[43 29]
 [39 33]]


# Subject A02

In [13]:
X, y, csp_objects = get_train_data(fno=2)
X_eval, y_eval = get_eval_data(fno=2,csp_objects=csp_objects)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 20 (2.2e-16 eps * 22 dim * 4.1e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channel

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


    Using tolerance 20 (2.2e-16 eps * 22 dim * 4e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 12 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 8.00, 12.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 event

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


    Using tolerance 9.6 (2.2e-16 eps * 22 dim * 2e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 12.00, 16.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 ev

...

<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 36 - 40 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 36.00, 40.00 Hz: -4.00, -4.00 dB



<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped


<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


In [14]:
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)

------Train---------
####### SVM#####
Accuracy Score:  0.7222222222222222
Cohen Kappa Score:  0.4444444444444444
ITR (bits per minute):  4.427844640515644
Confusion Matrix:  [[11  7]
 [ 3 15]]
##########LDA#########
Accuracy Score:  0.6666666666666666
Cohen Kappa Score:  0.33333333333333337
ITR (bits per minute):  2.4511249783653133
Confusion Matrix:  [[11  7]
 [ 5 13]]
------Test--------
#####SVM######
Accuracy Score:  0.4930555555555556
Cohen Kappa Score:  -0.01388888888888884
ITR (bits per minute):  0.004174599037647386
Confusion Matrix:  [[29 43]
 [30 42]]
#####LDA######
Accuracy Score:  0.4583333333333333
Cohen Kappa Score:  -0.08333333333333326
ITR (bits per minute):  0.15045515442089985
Confusion Matrix:  [[25 47]
 [31 41]]


In [15]:
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

--------Train--------
Accuracy Score:  0.6944444444444444
Cohen Kappa Score:  0.38888888888888884
ITR (bits per minute):  3.360710414545953
Confusion Matrix:  [[11  7]
 [ 4 14]]
--------Test------
Accuracy Score:  0.5694444444444444
Cohen Kappa Score:  0.13888888888888884
ITR (bits per minute):  0.4187990447866974
Confusion Matrix:  [[32 40]
 [22 50]]


# Subject A03

In [16]:
X, y, csp_objects = get_train_data(fno=3)
X_eval, y_eval = get_eval_data(fno=3,csp_objects=csp_objects)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 22 (2.2e-16 eps * 22 dim * 4.5e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channel

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 12 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 8.00, 12.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank fr

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 12.00, 16.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank 

...

<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 36 - 40 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 36.00, 40.00 Hz: -4.00, -4.00 dB



<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped


<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


In [17]:
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)

------Train---------
####### SVM#####
Accuracy Score:  0.7777777777777778
Cohen Kappa Score:  0.5555555555555556
ITR (bits per minute):  7.07386480474139
Confusion Matrix:  [[15  3]
 [ 5 13]]
##########LDA#########
Accuracy Score:  0.8055555555555556
Cohen Kappa Score:  0.6111111111111112
ITR (bits per minute):  8.679694384316315
Confusion Matrix:  [[15  3]
 [ 4 14]]
------Test--------
#####SVM######
Accuracy Score:  0.4930555555555556
Cohen Kappa Score:  -0.01388888888888884
ITR (bits per minute):  0.004174599037647386
Confusion Matrix:  [[ 7 65]
 [ 8 64]]
#####LDA######
Accuracy Score:  0.4930555555555556
Cohen Kappa Score:  -0.01388888888888884
ITR (bits per minute):  0.004174599037647386
Confusion Matrix:  [[10 62]
 [11 61]]


In [18]:
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

--------Train--------
Accuracy Score:  0.7222222222222222
Cohen Kappa Score:  0.4444444444444444
ITR (bits per minute):  4.427844640515644
Confusion Matrix:  [[12  6]
 [ 4 14]]
--------Test------
Accuracy Score:  0.4861111111111111
Cohen Kappa Score:  -0.02777777777777768
ITR (bits per minute):  0.01670000729103227
Confusion Matrix:  [[ 8 64]
 [10 62]]


# Subject A05 and A06

In [19]:
X, y, csp_objects = get_train_data(fno=4)
X_eval, y_eval = get_eval_data(fno=4,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 20 (2.2e-16 eps * 22 dim * 4e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels 

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 12 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 8.00, 12.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 9.3 (2.2e-16 eps * 22 dim * 1.9e+15  max singular va

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 12.00, 16.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 4.5 (2.2e-16 eps * 22 dim * 9.3e+14  max singular 

...

Creating RawArray with float64 data, n_channels=23, n_times=345415
    Range : 0 ... 345414 =      0.000 ...  1381.656 secs
Ready.
96 events found
Event IDs: [1 2]
96 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 96 events and 526 original time points ...
0 bad epochs dropped
Creating RawArray with float64 data, n_channels=22, n_times=345415
    Range : 0 ... 345414 =      0.000 ...  1381.656 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 36 - 40 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 36.00, 40.00 Hz: -4.00, -4.00 dB



<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=345415
    Range : 0 ... 345414 =      0.000 ...  1381.656 secs
Ready.
96 events found
Event IDs: [1 2]
96 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 96 events and 526 original time points ...
0 bad epochs dropped
------Train---------
####### SVM#####
Accuracy Score:  0.75
Cohen Kappa Score:  0.5
ITR (bits per minute):  5.661656266226015
Confusion Matrix:  [[14  4]
 [ 5 13]]
##########LDA#########
Accuracy Score:  0.6388888888888888
Cohen Kappa Score:  0.2777777777777778
ITR (bits per minute):  1.6919510610185395
Confusion Matrix:  [[13  5]
 [ 8 10]]
------Test--------
#####SVM######
Accuracy Score:  0.4895833333333333
Cohen Kappa Score:  -0.02083333333333326
ITR (bits per minute):  0.009393225394741522
Confusion Matrix:  [[ 2 46]
 [ 3 45]]
#####LDA######
Accuracy Score:  0.5104166666666666
Cohen Kappa Score:  0.02083333333333337
ITR (bits 

<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Accuracy Score:  0.6666666666666666
Cohen Kappa Score:  0.33333333333333337
ITR (bits per minute):  2.4511249783653133
Confusion Matrix:  [[13  5]
 [ 7 11]]
--------Test------
Accuracy Score:  0.4895833333333333
Cohen Kappa Score:  -0.02083333333333326
ITR (bits per minute):  0.009393225394741522
Confusion Matrix:  [[ 5 43]
 [ 6 42]]


In [20]:
X, y, csp_objects = get_train_data(fno=5)
X_eval, y_eval = get_eval_data(fno=5,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 21 (2.2e-16 eps * 22 dim * 4.4e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channel

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 12 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 8.00, 12.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 10 (2.2e-16 eps * 22 dim * 2.1e+15  max singular val

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 12.00, 16.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 4.7 (2.2e-16 eps * 22 dim * 9.7e+14  max singular 

...

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 36 - 40 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 36.00, 40.00 Hz: -4.00, -4.00 dB



<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
------Train---------
####### SVM#####
Accuracy Score:  0.8055555555555556
Cohen Kappa Score:  0.6111111111111112
ITR (bits per minute):  8.679694384316315
Confusion Matrix:  [[13  5]
 [ 2 16]]
##########LDA#########
Accuracy Score:  0.7777777777777778
Cohen Kappa Score:  0.5555555555555556
ITR (bits per minute):  7.07386480474139
Confusion Matrix:  [[13  5]
 [ 3 15]]
------Test--------
#####SVM######
Accuracy Score:  0.4930555555555556
Cohen Kappa Score:  -0.01388888888888884
ITR (bits per minute):  0.004174599037647386
Confusion Matrix:  [[69  3]
 [70  2]]
#####LDA######
Accuracy Score:  0.4722222222222222
Cohen Kappa Score:  

<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Accuracy Score:  0.7222222222222222
Cohen Kappa Score:  0.4444444444444444
ITR (bits per minute):  4.427844640515644
Confusion Matrix:  [[14  4]
 [ 6 12]]
--------Test------
Accuracy Score:  0.4791666666666667
Cohen Kappa Score:  -0.04166666666666674
ITR (bits per minute):  0.03758106191494637
Confusion Matrix:  [[69  3]
 [72  0]]


In [21]:
X, y, csp_objects = get_train_data(fno=6)
X_eval, y_eval = get_eval_data(fno=6,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 29 (2.2e-16 eps * 22 dim * 6e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels 

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 12 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 8.00, 12.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 15 (2.2e-16 eps * 22 dim * 3e+15  max singular value)
    Estimated rank (mag): 22
  

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 12.00, 16.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 7.1 (2.2e-16 eps * 22 dim * 1.4e+15  max singular value)
    Estimated rank (mag): 

...

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 36 - 40 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 36.00, 40.00 Hz: -4.00, -4.00 dB



<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
------Train---------
####### SVM#####
Accuracy Score:  0.75
Cohen Kappa Score:  0.5
ITR (bits per minute):  5.661656266226015
Confusion Matrix:  [[12  6]
 [ 3 15]]
##########LDA#########
Accuracy Score:  0.8611111111111112
Cohen Kappa Score:  0.7222222222222222
ITR (bits per minute):  12.56035503708892
Confusion Matrix:  [[16  2]
 [ 3 15]]
------Test--------
#####SVM######
Accuracy Score:  0.4652777777777778
Cohen Kappa Score:  -0.06944444444444442
ITR (bits per minute):  0.10444566385107823
Confusion Matrix:  [[26 46]
 [31 41]]
#####LDA######
Accuracy Score:  0.4861111111111111
Cohen Kappa Score:  -0.02777777777777768
ITR (bit

<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Accuracy Score:  0.7777777777777778
Cohen Kappa Score:  0.5555555555555556
ITR (bits per minute):  7.07386480474139
Confusion Matrix:  [[16  2]
 [ 6 12]]
--------Test------
Accuracy Score:  0.5138888888888888
Cohen Kappa Score:  0.02777777777777779
ITR (bits per minute):  0.016700007291030605
Confusion Matrix:  [[34 38]
 [32 40]]


In [22]:
X, y, csp_objects = get_train_data(fno=7)
X_eval, y_eval = get_eval_data(fno=7,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 19 (2.2e-16 eps * 22 dim * 4e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels 

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 12 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 8.00, 12.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank fr

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 12.00, 16.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 4.7 (2.2e-16 eps * 22 dim * 9.6e+14  max singular 

...

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 36 - 40 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 36.00, 40.00 Hz: -4.00, -4.00 dB



<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
------Train---------
####### SVM#####
Accuracy Score:  0.8333333333333334
Cohen Kappa Score:  0.6666666666666667
ITR (bits per minute):  10.499327350549375
Confusion Matrix:  [[17  1]
 [ 5 13]]
##########LDA#########
Accuracy Score:  0.7222222222222222
Cohen Kappa Score:  0.4444444444444444
ITR (bits per minute):  4.427844640515644
Confusion Matrix:  [[12  6]
 [ 4 14]]
------Test--------
#####SVM######
Accuracy Score:  0.4930555555555556
Cohen Kappa Score:  -0.01388888888888884
ITR (bits per minute):  0.004174599037647386
Confusion Matrix:  [[ 0 72]
 [ 1 71]]
#####LDA######
Accuracy Score:  0.5277777777777778
Cohen Kappa Score:

<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Accuracy Score:  0.6666666666666666
Cohen Kappa Score:  0.33333333333333337
ITR (bits per minute):  2.4511249783653133
Confusion Matrix:  [[13  5]
 [ 7 11]]
--------Test------
Accuracy Score:  0.5
Cohen Kappa Score:  0.0
ITR (bits per minute):  0.0
Confusion Matrix:  [[ 3 69]
 [ 3 69]]


In [23]:
X, y, csp_objects = get_train_data(fno=8)
X_eval, y_eval = get_eval_data(fno=8,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 26 (2.2e-16 eps * 22 dim * 5.2e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channel

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 12 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 8.00, 12.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 14 (2.2e-16 eps * 22 dim * 2.9e+15  max singular val

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 12.00, 16.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 7.5 (2.2e-16 eps * 22 dim * 1.5e+15  max singular 

...

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 36 - 40 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 36.00, 40.00 Hz: -4.00, -4.00 dB



<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
------Train---------
####### SVM#####
Accuracy Score:  0.8888888888888888
Cohen Kappa Score:  0.7777777777777778
ITR (bits per minute):  14.902249956730627
Confusion Matrix:  [[18  0]
 [ 4 14]]
##########LDA#########
Accuracy Score:  0.8888888888888888
Cohen Kappa Score:  0.7777777777777778
ITR (bits per minute):  14.902249956730627
Confusion Matrix:  [[18  0]
 [ 4 14]]
------Test--------
#####SVM######
Accuracy Score:  0.5277777777777778
Cohen Kappa Score:  0.05555555555555558
ITR (bits per minute):  0.06682583730054037
Confusion Matrix:  [[44 28]
 [40 32]]
#####LDA######
Accuracy Score:  0.5069444444444444
Cohen Kappa Score: 

<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Accuracy Score:  0.8888888888888888
Cohen Kappa Score:  0.7777777777777778
ITR (bits per minute):  14.902249956730627
Confusion Matrix:  [[18  0]
 [ 4 14]]
--------Test------
Accuracy Score:  0.5902777777777778
Cohen Kappa Score:  0.18055555555555558
ITR (bits per minute):  0.7093685992417309
Confusion Matrix:  [[50 22]
 [37 35]]


In [24]:
X, y, csp_objects = get_train_data(fno=9)
X_eval, y_eval = get_eval_data(fno=9,csp_objects=csp_objects)
pos, svm, lda, ss = train_mibif(X, y)
eval_mibif(pos, svm, lda, ss, X_eval, y_eval)
clf, ss = train_rf(X, y)
eval_rf(clf, ss, X_eval, y_eval)

Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 29 (2.2e-16 eps * 22 dim * 6e+15  max singular value)
    Estimated rank (mag): 22
    MAG: rank 22 computed from 22 data channels 

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 12 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 8.00, 12.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 16 (2.2e-16 eps * 22 dim * 3.4e+15  max singular val

<ipython-input-5-160d66fc0b01>:26: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.5, tmax=2.5)


Reducing data rank from 22 -> 22
Estimating covariance using EMPIRICAL
Done.
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 12 - 16 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 12.00, 16.00 Hz: -4.00, -4.00 dB

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 1251 original time points ...
0 bad epochs dropped
Computing rank from data with rank=None
    Using tolerance 8 (2.2e-16 eps * 22 dim * 1.6e+15  max singular va

...

Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
Creating RawArray with float64 data, n_channels=22, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 36 - 40 Hz

IIR filter parameters
---------------------
Chebyshev II bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 36.00, 40.00 Hz: -4.00, -4.00 dB



<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Creating RawArray with float64 data, n_channels=23, n_times=436266
    Range : 0 ... 436265 =      0.000 ...  1745.060 secs
Ready.
144 events found
Event IDs: [1 2]
144 matching events found
Applying baseline correction (mode: mean)
Not setting metadata
0 projection items activated
Loading data for 144 events and 526 original time points ...
0 bad epochs dropped
------Train---------
####### SVM#####
Accuracy Score:  0.7777777777777778
Cohen Kappa Score:  0.5555555555555556
ITR (bits per minute):  7.07386480474139
Confusion Matrix:  [[14  4]
 [ 4 14]]
##########LDA#########
Accuracy Score:  0.75
Cohen Kappa Score:  0.5
ITR (bits per minute):  5.661656266226015
Confusion Matrix:  [[12  6]
 [ 3 15]]
------Test--------
#####SVM######
Accuracy Score:  0.4722222222222222
Cohen Kappa Score:  -0.05555555555555558
ITR (bits per minute):  0.06682583730054037
Confusion Matrix:  [[62 10]
 [66  6]]
#####LDA######
Accuracy Score:  0.5069444444444444
Cohen Kappa Score:  0.01388888888888884
ITR (bits 

<ipython-input-5-160d66fc0b01>:59: RuntimeWarning: Cropping removes baseline period, setting epochs.baseline = None
  epochs = epochs.crop(tmin=0.0, tmax=2.0)


Accuracy Score:  0.6944444444444444
Cohen Kappa Score:  0.38888888888888884
ITR (bits per minute):  3.360710414545953
Confusion Matrix:  [[13  5]
 [ 6 12]]
--------Test------
Accuracy Score:  0.4583333333333333
Cohen Kappa Score:  -0.08333333333333326
ITR (bits per minute):  0.15045515442089985
Confusion Matrix:  [[60 12]
 [66  6]]
